# 03: Demonstração da API e do banco de dados

Este notebook sobe a API de inferência de verdade (`src/pdm/serving/api.py`, FastAPI +
Uvicorn) como um subprocesso, chama todos os seus endpoints via HTTP com `requests`,
exatamente como um cliente real faria, e depois lê o banco de dados de predições
diretamente com pandas, evidenciando a API e o banco de dados funcionando juntos (o
requisito "plus" do enunciado, reinterpretado em
[`docs/02_engenharia_requisitos.md`](../docs/02_engenharia_requisitos.md), seção 2.4).


In [1]:
import subprocess
import sys
import tempfile
import time
import json

import numpy as np
import pandas as pd
import requests

from pdm.config import load_config

cfg = load_config()
API_URL = f"http://{cfg.api['host'].replace('0.0.0.0', '127.0.0.1')}:{cfg.api['port']}"

# stdout/stderr go to a real file, not subprocess.PIPE: a pipe has a small
# OS buffer (64 KB on Linux), and nobody drains it while the API runs
# normally below. ssqueezepy's synchrosqueezed transform emits a
# RuntimeWarning (overflow/invalid value) to stderr on a meaningful
# fraction of real windows, and once the batch calls further down produce
# enough of those, the child process blocks on a full pipe mid-write and
# every request hangs indefinitely, invisible with a handful of windows but
# guaranteed once the drift demo below sends hundreds through /predict_batch
# with the real synchrosqueezed transform actually running.
api_log = tempfile.NamedTemporaryFile(mode="w+", prefix="pdm_api_", suffix=".log", delete=False)
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "pdm.serving.api:app",
     "--host", "127.0.0.1", "--port", str(cfg.api["port"])],
    stdout=api_log, stderr=subprocess.STDOUT,
)

for attempt in range(60):
    try:
        r = requests.get(f"{API_URL}/health", timeout=2)
        if r.status_code == 200:
            print("API no ar após", attempt + 1, "tentativa(s)")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    api_log.seek(0)
    print(api_log.read())
    raise RuntimeError("API não respondeu a tempo")


API no ar após 5 tentativa(s)


## `GET /health`

In [2]:
health = requests.get(f"{API_URL}/health").json()
print(json.dumps(health, indent=2, ensure_ascii=False))


{
  "status": "ok",
  "model_name": "hist_gradient_boosting",
  "classes": [
    "Classe A",
    "Classe B",
    "Classe C",
    "Classe D",
    "Classe E"
  ],
  "sensor_names": [
    "Dados_1",
    "Dados_2",
    "Dados_3"
  ],
  "window_len": 200,
  "sample_rate_hz": 10000,
  "bundle_created_at": "2026-09-13T02:00:54.604257Z"
}


## Uma janela real do conjunto de teste (nunca vista pelo modelo)

Em vez de dados sintéticos, usamos uma janela real do próprio conjunto de dados do case,
do split de teste reconstruído por `evaluate_bundle_on_holdout`, a mesma que embasa os
números do notebook 02.


In [3]:
from pdm.data.loader import load_sensor_dataset
from pdm.models.bundle import BUNDLE_FILENAME, ModelBundle
from pdm.models.pipeline import stratified_three_way_split_indices

bundle = ModelBundle.load(cfg.paths.models_dir / BUNDLE_FILENAME)
dataset = load_sensor_dataset(cfg)
_, _, idx_test = stratified_three_way_split_indices(dataset.labels, cfg.split, cfg.random_seed)

example_idx = int(idx_test[0])
window = {name: dataset.sensors[name][example_idx].tolist() for name in bundle.sensor_names}
true_label = dataset.labels[example_idx]
print(f"Janela real (índice {example_idx}), rótulo verdadeiro: {true_label}")


Janela real (índice 36037), rótulo verdadeiro: Classe E


## `POST /predict`

In [4]:
resp = requests.post(f"{API_URL}/predict", json={"sensors": window})
resp.raise_for_status()
prediction = resp.json()
print(json.dumps(prediction, indent=2, ensure_ascii=False))
print()
print("Acertou?", prediction["predicted_class"] == true_label)


{
  "predicted_class": "Classe E",
  "probabilities": {
    "Classe A": 2.9501493013736147e-06,
    "Classe B": 1.247025381192688e-05,
    "Classe C": 0.0007576995240051437,
    "Classe D": 0.0013379054432198212,
    "Classe E": 0.9978889746296618
  },
  "conformal_set": [
    "Classe E"
  ],
  "is_silent": false,
  "warnings": []
}

Acertou? True


## `POST /predict_batch`: várias janelas reais de uma vez

In [5]:
batch_idx = idx_test[1:6]
batch_windows = [
    {"sensors": {name: dataset.sensors[name][i].tolist() for name in bundle.sensor_names}}
    for i in batch_idx
]
batch_resp = requests.post(f"{API_URL}/predict_batch", json={"windows": batch_windows}).json()
pd.DataFrame(
    {
        "índice": batch_idx,
        "rótulo verdadeiro": dataset.labels[batch_idx],
        "predito": [r["predicted_class"] for r in batch_resp],
        "conjunto conformal": [r["conformal_set"] for r in batch_resp],
    }
)


,índice,rótulo verdadeiro,predito,conjunto conformal
0,45917,Classe E,Classe E,[Classe E]
1,5636,Classe B,Classe B,[Classe B]
2,49358,Classe A,Classe A,[Classe A]
3,43078,Classe D,Classe D,[Classe D]
4,16239,Classe E,Classe E,[Classe E]


## `POST /explain`: por que o modelo decidiu isso

In [6]:
explanation = requests.post(f"{API_URL}/explain", json={"sensors": window}).json()
print("Método:", explanation["method"])
pd.DataFrame(explanation["top_features"])


Método: shap


,feature,shap_value
0,Dados_3_cwt_scale4_energy_frac,1.035868
1,Dados_1_cwt_scale13_energy_frac,0.544494
2,Dados_2_cwt_scale0_energy_frac,0.528464
3,Dados_2_band_4000_4500hz,-0.460969
4,Dados_1_cwt_scale11_energy_frac,0.383991
5,Dados_2_band_2000_2500hz,0.358163
6,Dados_2_cwt_energy_log,0.356618
7,Dados_3_band_3000_3500hz,-0.350140
8,Dados_1_band_3000_3500hz,0.264276
9,Dados_1_cwt_scale4_energy_frac,0.256842


## `GET /audit`: a mesma auditoria do notebook 01, via API

Reexecuta a auditoria estatística e persiste o resultado no banco (`AuditRecord`).


In [7]:
audit_summary = requests.get(f"{API_URL}/audit", timeout=120).json()
print(json.dumps(audit_summary, indent=2, ensure_ascii=False))


{
  "n_windows": 50000,
  "class_balance_verdict": "perfectly balanced across classes; unlikely to occur naturally on a factory floor; treat as a curated/synthetic sample and re-evaluate under a realistic class prior before deployment (see evaluation/imbalance.py)",
  "recommended_sensors": [
    "Dados_1",
    "Dados_2",
    "Dados_3"
  ],
  "excluded_sensors": {
    "Dados_4": "stuck at (near-)constant value; no information, likely a dead/miswired sensor",
    "Dados_5": "statistically indistinguishable from white noise across classes; excluded"
  }
}


## `GET /drift`: comparando tráfego de produção real contra a referência de treino

Precisa de um número mínimo de predições registradas (`MIN_PRODUCTION_SAMPLES_FOR_DRIFT`
no `serving/api.py`) para não fabricar uma medição a partir de poucos pontos, já que o PSI fica
instável com poucas amostras por *bin* e dispara alarmes por puro ruído amostral, não por
drift real (ver `docs/09_mlops.md`). Por isso geramos aqui um lote real de produção do
tamanho mínimo exigido, não só um punhado de exemplos, usando `/predict_batch`, que
persiste o lote inteiro numa única transação (ver `serving/api.py`: gerar esse mesmo lote
com `/predict` em loop chegou a levar mais de 10 minutos, um HTTP round-trip e um commit
com fsync por janela).

Referência: *Population Stability Index* (PSI), métrica padrão de monitoramento de modelos de crédito adotada aqui para features de sensor (ver Siddiqi, 2006, *Credit Risk Scorecards*, para a formulação original), combinada com o teste de Kolmogorov-Smirnov por feature.


In [8]:
reload_resp = requests.post(f"{API_URL}/drift/reference/reload").json()
print("Referência recarregada:", reload_resp)

drift_batch_idx = idx_test[: 210]  # acima do MIN_PRODUCTION_SAMPLES_FOR_DRIFT configurado na API
drift_windows = [
    {"sensors": {name: dataset.sensors[name][i].tolist() for name in bundle.sensor_names}}
    for i in drift_batch_idx.tolist()
]
requests.post(f"{API_URL}/predict_batch", json={"windows": drift_windows})

drift_resp = requests.get(f"{API_URL}/drift")
print(drift_resp.status_code)
if drift_resp.status_code == 200:
    drift_summary = drift_resp.json()
    print(f"Alarmes: {drift_summary['n_alarms']} de {drift_summary['n_current']} janelas comparadas "
          f"contra {drift_summary['n_reference']} de referência")
    display(pd.DataFrame(drift_summary["top_features"]))
else:
    print(drift_resp.json())


Referência recarregada: {'reference_loaded': True, 'n_reference': 50000}


200
Alarmes: 0 de 200 janelas comparadas contra 50000 de referência


,feature,psi,ks_statistic,ks_p_value,alarm
0,Dados_1_band_4000_4500hz,0.166451,0.15340,0.000149,False
1,Dados_2_peak1_freq_hz,0.147641,0.03222,0.981811,False
2,Dados_3_peak2_freq_hz,0.144062,0.09950,0.036227,False
3,Dados_1_band_3500_4000hz,0.135007,0.13568,0.001175,False
4,Dados_1_cwt_scale0_energy_frac,0.134535,0.11382,0.010586,False
5,Dados_2_peak2_freq_hz,0.126363,0.03096,0.988078,False
6,Dados_3_cwt_dominant_scale_idx,0.123853,0.03460,0.964420,False
7,Dados_2_peak_to_peak,0.123460,0.12872,0.002468,False
8,Dados_2_std,0.121627,0.12650,0.003101,False
9,Dados_2_peak,0.118509,0.10218,0.029141,False


**Interpretação**: como este lote de "produção" foi extraído do próprio conjunto de teste
(logo, da mesma distribuição da referência de treino), o esperado, e o que valida a
implementação, é um número de alarmes baixo, não zero cravado (o próprio conjunto de
teste já é uma amostra finita). Um número de alarmes alto aqui indicaria um bug no monitor
de drift, não drift real; isso foi verificado na prática (ver `docs/09_mlops.md`) e é por
isso que o tamanho mínimo de lote existe.


## Provocando um alarme de drift real

O teste acima usa dados da mesma distribuição do treino, por isso o número baixo de alarmes é o resultado esperado, não uma ausência de teste. Para confirmar que o monitor realmente distingue drift de ruído amostral, simulamos aqui um defeito de sensor plausível em campo, um ganho de amplificação fora de calibração em `Dados_1`, e verificamos que o mesmo endpoint `GET /drift`, com o mesmo limiar, agora dispara alarmes de verdade.


In [9]:
drifted_idx = idx_test[210:420]  # lote diferente do usado no teste "sem drift" acima
drifted_windows = []
for i in drifted_idx.tolist():
    sensors = {name: dataset.sensors[name][i].tolist() for name in bundle.sensor_names}
    sensors["Dados_1"] = (np.array(sensors["Dados_1"]) * 1.8).tolist()  # ganho fora de calibração
    drifted_windows.append({"sensors": sensors})

requests.post(f"{API_URL}/predict_batch", json={"windows": drifted_windows})

drift_resp2 = requests.get(f"{API_URL}/drift")
drift_summary2 = drift_resp2.json()
print(f"Alarmes: {drift_summary2['n_alarms']} de {drift_summary2['n_current']} janelas comparadas "
      f"contra {drift_summary2['n_reference']} de referência")
display(pd.DataFrame(drift_summary2["top_features"]))


Alarmes: 7 de 200 janelas comparadas contra 50000 de referência


,feature,psi,ks_statistic,ks_p_value,alarm
0,Dados_1_rms,11.347589,0.96372,4.729534e-287,True
1,Dados_1_peak,10.465358,0.90456,1.929867e-203,True
2,Dados_1_std,10.371773,0.95630,5.712629e-271,True
3,Dados_1_peak_to_peak,9.582312,0.92956,1.041293e-229,True
4,Dados_1_spectral_energy_log,9.489225,0.90268,9.444426e-202,True
5,Dados_1_cwt_energy_log,9.489225,0.93570,1.365023e-237,True
6,Dados_1_mean,0.965489,0.41888,3.745304e-32,True
7,Dados_2_band_2000_2500hz,0.117971,0.07750,1.737336e-01,False
8,Dados_3_band_3000_3500hz,0.101175,0.10104,3.199087e-02,False
9,Dados_2_band_1500_2000hz,0.098418,0.08440,1.108255e-01,False


**Interpretação**: com o mesmo endpoint, o mesmo limiar e um lote de tamanho comparável ao anterior, o número de alarmes sobe de forma clara assim que a distribuição de entrada realmente muda: o monitor não está "sempre ligado" nem "sempre desligado", ele responde à magnitude real do desvio. É esse comportamento (discriminar drift real de ruído amostral) que sustenta o gatilho de retrain descrito em `docs/09_mlops.md`.


## Lendo o banco de dados diretamente

Tudo que a API produziu nesta sessão, predições, auditoria, relatório de drift, está
persistido e consultável com SQL comum, não só através da própria API.


In [10]:
import sqlalchemy as sa

engine = sa.create_engine(cfg.api["database_url"])
predictions_df = pd.read_sql("SELECT id, created_at, model_name, predicted_class, is_silent FROM predictions "
                              "ORDER BY id DESC LIMIT 10", engine)
predictions_df


,id,created_at,model_name,predicted_class,is_silent
0,1069,2026-09-13 15:24:12.267249,hist_gradient_boosting,Classe B,0
1,1068,2026-09-13 15:24:12.267248,hist_gradient_boosting,Classe E,0
2,1067,2026-09-13 15:24:12.267247,hist_gradient_boosting,Classe C,0
3,1066,2026-09-13 15:24:12.267246,hist_gradient_boosting,Classe D,0
4,1065,2026-09-13 15:24:12.267245,hist_gradient_boosting,Classe E,0
5,1064,2026-09-13 15:24:12.267244,hist_gradient_boosting,Classe E,1
6,1063,2026-09-13 15:24:12.267243,hist_gradient_boosting,Classe A,0
7,1062,2026-09-13 15:24:12.267242,hist_gradient_boosting,Classe E,0
8,1061,2026-09-13 15:24:12.267241,hist_gradient_boosting,Classe A,0
9,1060,2026-09-13 15:24:12.267239,hist_gradient_boosting,Classe E,0


In [11]:
print("Total de predições registradas:", pd.read_sql("SELECT COUNT(*) as n FROM predictions", engine)["n"].iloc[0])
print("Total de auditorias registradas:", pd.read_sql("SELECT COUNT(*) as n FROM audits", engine)["n"].iloc[0])
print("Total de relatórios de drift:", pd.read_sql("SELECT COUNT(*) as n FROM drift_reports", engine)["n"].iloc[0])


Total de predições registradas: 1069
Total de auditorias registradas: 3
Total de relatórios de drift: 3


## Encerrando o servidor de demonstração

In [12]:
proc.terminate()
try:
    proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    proc.kill()
api_log.close()
print("API encerrada.")


API encerrada.


## Conclusão

A API expõe o mesmo pipeline testado nos notebooks 01 e 02 como um serviço HTTP com
persistência real em banco de dados: previsão, auditoria e monitoramento de drift não
são funcionalidades separadas, são três visões do mesmo sistema. Ver
[`docs/03_arquitetura.md`](../docs/03_arquitetura.md) para os diagramas de arquitetura e
sequência, e `docker/docker-compose.yml` para a versão contêinerizada com Postgres.
